In [2]:
from typing import Dict, Any, List

In [16]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

In [17]:
load_dotenv()

# Initialize the client exactly like the lab setup
# (Will pull OPENAI_API_KEY natively from your .env file)
client = OpenAI()

In [24]:
# =====================================================================
# 1. THE INPUT DATA
# =====================================================================
job_description = """
Role: Financial Account Manager
Requirements:
- 3+ years managing high-throughput infrastructure.
- Deep expertise with financial cloud infrastructure and security.
- Experience coaching junior engineers and working across teams.
"""

candidate_resume = """
Name: Jordan Vance
Experience:
- 3 years working for a cloud consultancy.
- Automated CI/CD pipelines, reducing deploy times by 40%.
- Conducted weekly architecture reviews and mentored 4 junior engineers.
- Highly analytical, and people friendly.
"""

# =====================================================================
# 2. THE MOCK CLIENT (Updated for Orchestrator Pattern)
# =====================================================================
class MockLLMClient:
    """Simulates LLM responses for the Orchestrator and its Workers."""
    def generate(self, system_prompt: str, user_prompt: str) -> str:
        prompt_lower = system_prompt.lower()
        
        # Worker 1's simulated response
        if "extractor" in prompt_lower:
            return json.dumps({
                "requirements": ["3+ years high-throughput infra", "Financial cloud security", "Mentorship"]
            }, indent=2)
            
        # Worker 2's simulated response
        elif "evaluator" in prompt_lower:
            return json.dumps({
                "strengths": ["3 years cloud experience", "Mentorship (4 junior engineers)"],
                "gaps": ["No explicit mention of 'financial' cloud security"],
                "score": "75%"
            }, indent=2)
            
        # Orchestrator's final synthesis response
        elif "orchestrator" in prompt_lower:
            return (
                "Final Recommendation for Jordan Vance:\n"
                "Jordan is a strong technical fit with proven mentorship skills. "
                "However, the Orchestrator flagged a missing requirement regarding financial security. "
                "Recommendation: Advance to technical interview, but explicitly probe on FinTech security background."
            )
            
        return '{"error": "Unknown prompt"}'

# =====================================================================
# 3. THE WORKERS
# =====================================================================
class RequirementsExtractor:
    def __init__(self, client):
        self.client = client
        self.system_prompt = "You are an extractor. Convert the job description into a JSON list of core requirements."

    def execute(self, jd_text: str) -> dict:
        print("   [Worker 1] Extracting core requirements from Job Description...")
        raw = self.client.generate(self.system_prompt, jd_text)
        return json.loads(raw)


class CandidateEvaluator:
    def __init__(self, client):
        self.client = client
        self.system_prompt = "You are an evaluator. Compare the resume against the requirements and return JSON strengths/gaps."

    def execute(self, resume_text: str, requirements: dict) -> dict:
        print("   [Worker 2] Cross-referencing candidate experience with extracted requirements...")
        user_prompt = f"Resume: {resume_text}\nReqs: {requirements}"
        raw = self.client.generate(self.system_prompt, user_prompt)
        return json.loads(raw)

# =====================================================================
# 4. THE ORCHESTRATOR
# =====================================================================
class OrchestratorAgent:
    def __init__(self, client):
        self.client = client
        # The orchestrator "owns" the workers
        self.extractor = RequirementsExtractor(client)
        self.evaluator = CandidateEvaluator(client)
        self.system_prompt = "You are the Lead Orchestrator. Synthesize the workers' findings into a final hiring recommendation."

    def run_hiring_pipeline(self, jd: str, resume: str) -> str:
        print("⚙️ [Orchestrator] Initiating analysis pipeline...")
        
        # Step 1: Orchestrator delegates task 1 to the Extractor
        reqs_json = self.extractor.execute(jd)
        
        # Step 2: Orchestrator passes the output of task 1 and the resume to task 2
        evaluation_json = self.evaluator.execute(resume, reqs_json)
        
        # Step 3: Orchestrator synthesizes all the sub-tasks into a final decision
        print("⚙️ [Orchestrator] Synthesizing worker reports into final recommendation...\n")
        user_prompt = f"Requirements: {reqs_json}\nEvaluation: {evaluation_json}"
        final_decision = self.client.generate(self.system_prompt, user_prompt)
        
        return final_decision

# =====================================================================
# 5. EXECUTE
# =====================================================================
client = MockLLMClient()
orchestrator = OrchestratorAgent(client)

final_report = orchestrator.run_hiring_pipeline(job_description, candidate_resume)

print("="*50)
print(final_report)
print("="*50)

⚙️ [Orchestrator] Initiating analysis pipeline...
   [Worker 1] Extracting core requirements from Job Description...
   [Worker 2] Cross-referencing candidate experience with extracted requirements...
⚙️ [Orchestrator] Synthesizing worker reports into final recommendation...

Final Recommendation for Jordan Vance:
Jordan is a strong technical fit with proven mentorship skills. However, the Orchestrator flagged a missing requirement regarding financial security. Recommendation: Advance to technical interview, but explicitly probe on FinTech security background.


[ IN ]
    Job Description 
    Candidate Resume
         │
         ▼
[  LEAD ORCHESTRATOR ]
         │
         ├── 1. Delegates tasks ───► [  Worker 1: Extractor ]
         │                               (Extracts requirements from JD)
         │
         ├── 2. Receives data   ◄─── [  Output: Requirements JSON ]
         │
         ├── 3. Delegates tasks ───► [  Worker 2: Evaluator ]
         │                               (Scores Resume against Requirements)
         │
         ├── 4. Receives data   ◄─── [  Output: Evaluation JSON ]
         │
         ▼
[  LEAD ORCHESTRATOR ]
  (Synthesizes Worker 1 & 2 data)
         │
         ▼
[ OUT ]
   Final Hiring Recommendation